In [ ]:
# uv pip install mlx-tune
# pip install torchvision   # required for Qwen3-VL processor

# uv pip install 'mlx-tune[audio]'
# brew install ffmpeg

In [ ]:
from mlx_tune import FastLanguageModel, SFTTrainer, SFTConfig
from datasets import load_dataset

/Users/linghuang/miniconda3/envs/llm/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [ ]:
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name="mlx-community/Llama-3.2-1B-Instruct-4bit",
    max_seq_length=2048,
    load_in_4bit=True,
)

/Users/linghuang/miniconda3/envs/llm/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Fetching 6 files: 100%|██████████| 6/6 [00:10<00:00,  1.75s/it]


In [ ]:
model = FastLanguageModel.get_peft_model(
    model,
    r=16,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj"],
    lora_alpha=16,
)

LoRA configuration set: rank=16, alpha=16, modules=['q_proj', 'k_proj', 'v_proj', 'o_proj'], dropout=0.0


In [ ]:
dataset = load_dataset("yahma/alpaca-cleaned", split="train[:100]")

Generating train split: 100%|██████████| 51760/51760 [00:00<00:00, 227468.61 examples/s]


### SFTTrainer

In [ ]:
trainer = SFTTrainer(
    model=model,
    train_dataset=dataset,
    tokenizer=tokenizer,
    args=SFTConfig(
        output_dir="outputs",
        per_device_train_batch_size=2,
        learning_rate=2e-4,
        max_steps=50,
    ),
)
trainer.train()


Trainer initialized:
  Output dir: outputs
  Adapter path: outputs/adapters
  Learning rate: 0.0002
  Iterations: 50
  Batch size: 2
  LoRA r=16, alpha=16
  Native training: True
  LR scheduler: cosine
  Grad checkpoint: False
Starting Fine-Tuning

[Using Native MLX Training]

Applying LoRA adapters...
Applying LoRA to 16 layers: {'rank': 16, 'scale': 1.0, 'dropout': 0.0, 'keys': ['self_attn.k_proj', 'self_attn.o_proj', 'self_attn.q_proj', 'self_attn.v_proj']}
✓ LoRA applied successfully to 16 layers
  Trainable LoRA parameters: 128
Preparing training data...
  Detected format: alpaca
✓ Prepared 100 training samples
  Saved to: outputs/train.jsonl
✓ Created validation set (copied from train)

Training configuration:
  Iterations: 50
  Batch size: 2
  Learning rate: 0.0002
  LR scheduler: cosine
  Grad checkpoint: True
  Adapter file: outputs/adapters/adapters.safetensors

Loaded 100 training samples, 100 validation samples
Starting training loop...
Starting training..., iters: 50


Calculating loss...: 100%|██████████| 25/25 [00:12<00:00,  2.03it/s]

Iter 1: Val loss 1.961, Val took 12.304s


Iter 10: Train loss 1.493, Learning Rate 1.844e-04, It/sec 0.700, Tokens/sec 222.360, Trained Tokens 3177, Peak mem 1.973 GB
Iter 20: Train loss 1.330, Learning Rate 1.368e-04, It/sec 0.796, Tokens/sec 229.066, Trained Tokens 6054, Peak mem 1.973 GB
Iter 30: Train loss 1.185, Learning Rate 7.513e-05, It/sec 0.515, Tokens/sec 217.201, Trained Tokens 10271, Peak mem 2.040 GB
Iter 40: Train loss 1.470, Learning Rate 2.295e-05, It/sec 0.615, Tokens/sec 236.137, Trained Tokens 14111, Peak mem 2.215 GB


Calculating loss...: 100%|██████████| 25/25 [00:12<00:00,  1.95it/s]

Iter 50: Val loss 1.187, Val took 12.834s


Iter 50: Train loss 1.383, Learning Rate 1.973e-07, It/sec 0.511, Tokens/sec 191.437, Trained Tokens 17856, Peak mem 2.215 GB
Saved final weights to outputs/adapters/adapters.safetensors.
  Adapter config saved to: outputs/adapters/adapter_config.json

Training Complete!
  Adapters saved to: outputs/adapters


{'status': 'success', 'adapter_path': 'outputs/adapters'}

In [ ]:
model.save_pretrained("lora_model")            # LoRA adapters only
model.save_pretrained_merged("merged", tokenizer)   # fused MLX model
# GGUF export is NOT supported for quantized base models (mlx-lm limitation).
# To get a GGUF, fuse first then re-convert with llama.cpp:
#   mlx_lm.fuse --model mlx-community/Llama-3.2-1B-Instruct-4bit \
#               --adapter-path outputs/adapters --save-path fused_model --de-quantize
#   python llama.cpp/convert_hf_to_gguf.py fused_model --outfile model/model.gguf --outtype q4_k_m

In [ ]:
from mlx_lm import load, generate

model_inf, tokenizer_inf = load("merged")

TEMPLATE = (
    "Below is an instruction that describes a task. "
    "Write a response that appropriately completes the request.\n\n"
    "### Instruction:\n{instruction}\n\n### Response:\n"
)

prompts = [
    "Give three tips for staying healthy.",
    "What are the three primary colors?",
    "Explain what machine learning is in simple terms.",
]

for instruction in prompts:
    print(f"\n{'='*60}\nInstruction: {instruction}\nResponse:")
    response = generate(
        model_inf,
        tokenizer_inf,
        prompt=TEMPLATE.format(instruction=instruction),
        max_tokens=256,
        verbose=False,
    )
    print(response)


Instruction: Give three tips for staying healthy.
Response:
Here are three tips for staying healthy:

1. **Stay Hydrated**: Drinking plenty of water throughout the day is essential for maintaining good health. Aim to drink at least eight glasses of water a day, and make sure to drink plenty of water before, during, and after exercise. Staying hydrated helps to flush out toxins, boost energy levels, and support overall health.

2. **Eat a Balanced Diet**: A well-balanced diet that includes a variety of fruits, vegetables, whole grains, lean proteins, and healthy fats is crucial for maintaining good health. Aim to eat at least five servings of fruits and vegetables a day, and include a source of protein in every meal. A balanced diet helps to provide the necessary nutrients for growth and development, as well as to support overall health.

3. **Get Enough Sleep**: Getting enough sleep is essential for maintaining good health. Aim to get at least seven hours of sleep a night, and establi

### Vision Model - VLMSFTTrainer

In [ ]:
from mlx_tune import FastVisionModel, UnslothVisionDataCollator, VLMSFTTrainer
from mlx_tune.vlm import VLMSFTConfig

model, processor = FastVisionModel.from_pretrained(
"mlx-community/Qwen3.5-0.8B-bf16",
)

model = FastVisionModel.get_peft_model(
    model,
    finetune_vision_layers=True,    
    finetune_language_layers=True,
    r=16, lora_alpha=16,
)

Loading VLM: mlx-community/Qwen3.5-0.8B-bf16


Fetching 10 files: 100%|██████████| 10/10 [00:00<00:00, 127875.12it/s]
[transformers] The `use_fast` parameter is deprecated and will be removed in a future version. Use `backend="torchvision"` instead of `use_fast=True`, or `backend="pil"` instead of `use_fast=False`.
Fetching 10 files: 100%|██████████| 10/10 [00:00<00:00, 218453.33it/s]


#trainable params: 6.38976 M || all params: 752.392448 M || trainable%: 0.849%
LoRA configured: r=16, alpha=16, modules=['q_proj', 'k_proj', 'v_proj', 'o_proj', 'gate_proj', 'up_proj', 'down_proj']


In [ ]:
from datasets import load_dataset
from PIL import Image

vlm_dataset = load_dataset("unsloth/Radiology_mini", split="train")

# Inspect actual schema
print("Columns:", vlm_dataset.column_names)
sample = vlm_dataset[0]
for k, v in sample.items():
    if k != "image":
        print(f"  {k!r}: {str(v)[:120]}")

Columns: ['image', 'image_id', 'caption', 'cui']
  'image_id': ROCOv2_2023_train_054311
  'caption': Panoramic radiography shows an osteolytic lesion in the right posterior maxilla with resorption of the floor of the maxi
  'cui': ['C1306645', 'C0037303']


In [ ]:
cols = vlm_dataset.column_names

def to_messages(row):
    img = row["image"].convert("RGB")
    if "messages" in cols:
        # Already in conversation format — just ensure images key exists
        return {"messages": row["messages"], "images": [img]}
    if "conversations" in cols:
        # ShareGPT format: [{"from": "human", "value": "..."}, {"from": "gpt", "value": "..."}]
        convs = row["conversations"]
        human = next(c["value"] for c in convs if c["from"] in ("human", "user"))
        gpt   = next(c["value"] for c in convs if c["from"] in ("gpt", "assistant"))
        question, answer = human, gpt
    elif "question" in cols and "answer" in cols:
        question, answer = row["question"], row["answer"]
    elif "caption" in cols:
        question, answer = "Describe this image.", row["caption"]
    elif "text" in cols:
        question, answer = "Describe this image.", row["text"]
    else:
        # Fallback: use first non-image string column as answer
        text_col = next(c for c in cols if c != "image")
        question, answer = "Describe this image.", row[text_col]

    return {
        "messages": [
            {"role": "user", "content": [
                {"type": "image"},
                {"type": "text", "text": question}
            ]},
            {"role": "assistant", "content": [
                {"type": "text", "text": answer}
            ]}
        ],
        "images": [img]
    }

vlm_dataset = vlm_dataset.map(to_messages, remove_columns=vlm_dataset.column_names)
print(f"Ready: {len(vlm_dataset)} samples")
print(vlm_dataset[0]["messages"])

# # LaTeX OCR — document/math image understanding
# dataset = load_dataset("unsloth/LaTeX_OCR", split="train[:200]")

Map: 100%|██████████| 1978/1978 [03:01<00:00, 10.88 examples/s] 

Ready: 1978 samples
[{'role': 'user', 'content': [{'type': 'image', 'text': None}, {'type': 'text', 'text': 'Describe this image.'}]}, {'role': 'assistant', 'content': [{'type': 'text', 'text': 'Panoramic radiography shows an osteolytic lesion in the right posterior maxilla with resorption of the floor of the maxillary sinus (arrows).'}]}]


In [ ]:
FastVisionModel.for_training(model)
trainer = VLMSFTTrainer(
    model=model,
    tokenizer=processor,
    data_collator=UnslothVisionDataCollator(model, processor),
    train_dataset=vlm_dataset,
    args=VLMSFTConfig(max_steps=10, learning_rate=2e-4),
)
trainer.train()

/Users/linghuang/miniconda3/envs/llm/lib/python3.10/site-packages/mlx_tune/vlm.py:1337: UserWarning: VLM training with batch_size=2 is not recommended. Images of different sizes produce different vision token counts, which causes shape mismatches. Using batch_size=1 instead.
  warnings.warn(


Starting VLM Fine-Tuning
Training with mlx-vlm Dataset


Training: 100%|██████████| 10/10 [12:55<00:00, 77.58s/it, loss=3.3438, avg=3.2437]



Training complete! Average loss: 3.2437
Adapters saved to: outputs/adapters


In [ ]:
# Option A — save LoRA adapters only (small, ~MB)
# model.save_pretrained("vlm_lora", processor)

# Option B — fuse adapters into base weights (self-contained model)
model.save_pretrained_merged("vlm_merged", processor)

Fused 96 LoRA layers into base model
Merged model saved to vlm_merged


In [ ]:
from mlx_tune import FastVisionModel, UnslothVisionDataCollator, VLMSFTTrainer

model_inf, processor_inf = FastVisionModel.from_pretrained(
    "mlx-community/Qwen3.5-0.8B-bf16",
    adapter_path="outputs/adapters",  
)

Loading VLM: mlx-community/Qwen3.5-0.8B-bf16


Fetching 10 files: 100%|██████████| 10/10 [00:00<00:00, 45491.37it/s]


#trainable params: 10.822656 M || all params: 752.392448 M || trainable%: 1.438%


[transformers] The `use_fast` parameter is deprecated and will be removed in a future version. Use `backend="torchvision"` instead of `use_fast=True`, or `backend="pil"` instead of `use_fast=False`.
Fetching 10 files: 100%|██████████| 10/10 [00:00<00:00, 109798.53it/s]


In [ ]:
from mlx_vlm.prompt_utils import apply_chat_template
from mlx_vlm.generate import generate

FastVisionModel.for_inference(model_inf)

sample = vlm_dataset[0]
image = sample["images"][0]
question = sample["messages"][0]["content"][1]["text"]
image.save("/tmp/test_image.jpg")

prompt = apply_chat_template(
    processor_inf,
    model_inf.model.config,
    question,
    num_images=1,
)

response = generate(
    model_inf.model,
    processor_inf,
    prompt=prompt,
    image=["/tmp/test_image.jpg"],
    max_tokens=256,
    verbose=True,
)
print(response.text)

### TTSSFTTrainer

In [1]:
from mlx_tune import FastTTSModel, TTSSFTTrainer, TTSSFTConfig, TTSDataCollator
from datasets import load_dataset, Audio

/Users/linghuang/miniconda3/envs/llm/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
model, tokenizer = FastTTSModel.from_pretrained(
"mlx-community/orpheus-3b-0.1-ft-bf16"
)
model = FastTTSModel.get_peft_model(model, r=16, lora_alpha=16)

Loading TTS model: mlx-community/orpheus-3b-0.1-ft-bf16


Fetching 7 files: 100%|██████████| 7/7 [01:10<00:00, 10.14s/it]


Loading audio codec: mlx-community/snac_24khz


Fetching 2 files: 100%|██████████| 2/2 [00:04<00:00,  2.22s/it]


TTS model loaded: mlx-community/orpheus-3b-0.1-ft-bf16
Audio codec: snac (sample rate: 24000Hz)
LoRA configuration set: rank=16, alpha=16, modules=['q_proj', 'k_proj', 'v_proj', 'o_proj', 'gate_proj', 'up_proj', 'down_proj'], dropout=0.0


In [4]:
dataset = load_dataset("hf-internal-testing/librispeech_asr_demo", "clean", split="validation[:100]")
dataset = dataset.cast_column("audio", Audio(sampling_rate=24000))

Generating validation split: 100%|██████████| 73/73 [00:00<00:00, 709.18 examples/s]


In [ ]:
trainer = TTSSFTTrainer(
    model=model, tokenizer=tokenizer,
    data_collator=TTSDataCollator(model, tokenizer),
    train_dataset=dataset,
    args=TTSSFTConfig(output_dir="./tts_output", max_steps=1),
)
trainer.train()

Starting TTS Fine-Tuning
Applying LoRA to 28 layers: {'rank': 16, 'scale': 1.0, 'dropout': 0.0, 'keys': ['self_attn.q_proj', 'self_attn.k_proj', 'self_attn.v_proj', 'self_attn.o_proj', 'mlp.gate_proj', 'mlp.up_proj', 'mlp.down_proj']}
LoRA applied: 392 trainable parameter groups across 28 layers
  Model: mlx-community/orpheus-3b-0.1-ft-bf16
  Dataset: 73 samples
  Total steps: 60
  Batch size: 1
  Gradient accumulation: 4
  Learning rate: 0.0002


Training:   0%|          | 0/60 [00:00<?, ?it/s]